# Semana 9: Transformers: Atención Multi-Head y Positional Encoding
## Notebook Conceptual (NB1) – Implementación de Self-Attention

**Propósito:** Comprender la arquitectura del transformer desde cero, implementando self-attention y positional encoding.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Calcular self-attention: matrices Q, K, V y softmax sobre puntuaciones.
- Implementar multi-head attention concatenando cabezas.
- Añadir positional encoding (seno/coseno) a los embeddings.
- Construir un bloque transformer completo (atención + feed-forward + residual + layer norm).
- Probar la propagación hacia adelante con datos dummy.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos PyTorch.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Configuración de PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Semillas para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

print("\nLibrerías importadas correctamente.")

---
## 1. Datos Dummy: Secuencia de Embeddings

Generamos una secuencia de vectores aleatorios que simulan los embeddings de palabras de una frase.

In [ ]:
# Parámetros
seq_len = 6      # longitud de la secuencia (número de palabras)
embed_dim = 8    # dimensión de los embeddings

# Generar embeddings aleatorios
X = torch.randn(seq_len, embed_dim)
print(f"Forma de X: {X.shape}")
print("Matriz de embeddings X:")
print(X.numpy().round(3))

---
## 2. Self-Attention

### 2.1. Implementación básica de self-attention

In [ ]:
def self_attention_basic(X, d_k=None):
    """
    Implementación básica de self-attention.
    X: (seq_len, embed_dim)
    d_k: dimensión de los vectores query/key (si None, se usa embed_dim)
    """
    seq_len, embed_dim = X.shape
    if d_k is None:
        d_k = embed_dim
    
    # Inicializar matrices de pesos aleatorias (simulando aprendizaje)
    W_q = torch.randn(embed_dim, d_k)
    W_k = torch.randn(embed_dim, d_k)
    W_v = torch.randn(embed_dim, d_k)
    
    # Calcular Q, K, V
    Q = X @ W_q  # (seq_len, d_k)
    K = X @ W_k  # (seq_len, d_k)
    V = X @ W_v  # (seq_len, d_k)
    
    # Calcular puntuaciones de atención
    scores = Q @ K.T  # (seq_len, seq_len)
    
    # Escalar
    scores = scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    
    # Aplicar softmax
    attn_weights = F.softmax(scores, dim=-1)  # (seq_len, seq_len)
    
    # Salida ponderada
    output = attn_weights @ V  # (seq_len, d_k)
    
    return output, attn_weights

output, attn_weights = self_attention_basic(X)
print("Matriz de atención (pesos):")
print(attn_weights.numpy().round(3))
print(f"\nForma de la salida: {output.shape}")

In [ ]:
# Visualizar la matriz de atención
plt.figure(figsize=(8, 6))
sns.heatmap(attn_weights.numpy(), annot=True, fmt='.2f', cmap='Blues')
plt.xlabel('Posición Key')
plt.ylabel('Posición Query')
plt.title('Matriz de Atención Self-Attention')
plt.show()

---
## 3. Multi-Head Attention

Implementamos atención multi-head que ejecuta varias atenciones en paralelo y concatena los resultados.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim debe ser divisible por num_heads"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        # Proyecciones lineales para Q, K, V
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, x):
        batch_size, seq_len, embed_dim = x.shape
        
        # Proyectar y dividir en cabezas
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Calcular atención para todas las cabezas
        scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.head_dim, dtype=torch.float32))
        attn_weights = F.softmax(scores, dim=-1)
        
        # Aplicar atención a V
        context = torch.matmul(attn_weights, V)  # (batch_size, num_heads, seq_len, head_dim)
        
        # Concatenar cabezas
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)
        
        # Proyección final
        output = self.W_o(context)
        
        return output, attn_weights

# Crear instancia y probar
mha = MultiHeadAttention(embed_dim=8, num_heads=2)

# Añadir dimensión de batch
X_batch = X.unsqueeze(0)  # (1, seq_len, embed_dim)
output_mha, attn_mha = mha(X_batch)

print(f"Forma de la salida multi-head: {output_mha.shape}")
print(f"Forma de los pesos de atención: {attn_mha.shape}")  # (batch, num_heads, seq_len, seq_len)

In [ ]:
# Visualizar atención para cada cabeza
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for head in range(2):
    sns.heatmap(attn_mha[0, head].detach().numpy(), annot=True, fmt='.2f', cmap='Blues', ax=axes[head])
    axes[head].set_title(f'Cabeza {head+1}')
    axes[head].set_xlabel('Posición Key')
    axes[head].set_ylabel('Posición Query')

plt.tight_layout()
plt.show()

---
## 4. Positional Encoding

Añadimos información de posición a los embeddings usando funciones seno y coseno.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=100):
        super().__init__()
        
        # Crear matriz de positional encodings
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        
        # Fórmula: PE(pos, 2i) = sin(pos/10000^(2i/embed_dim))
        #          PE(pos, 2i+1) = cos(pos/10000^(2i/embed_dim))
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-np.log(10000.0) / embed_dim))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Registrar como buffer (no entrenable)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        return x + self.pe[:x.size(1), :]

# Probar positional encoding
pos_encoder = PositionalEncoding(embed_dim=8, max_len=20)

# Añadir positional encoding a los embeddings
X_with_pos = pos_encoder(X_batch)

print("Embedding original (primer token):")
print(X_batch[0, 0].numpy().round(3))
print("\nEmbedding con positional encoding (primer token):")
print(X_with_pos[0, 0].numpy().round(3))

In [ ]:
# Visualizar positional encodings
plt.figure(figsize=(12, 6))
plt.imshow(pos_encoder.pe.numpy(), cmap='viridis', aspect='auto')
plt.colorbar(label='Valor')
plt.xlabel('Dimensión del embedding')
plt.ylabel('Posición')
plt.title('Positional Encoding (seno/coseno)')
plt.show()

---
## 5. Feed-Forward Network (FFN)

Capa feed-forward que se aplica a cada posición independientemente.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim, ff_dim, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, ff_dim)
        self.fc2 = nn.Linear(ff_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        return self.fc2(self.dropout(self.relu(self.fc1(x))))

ffn = FeedForward(embed_dim=8, ff_dim=16)
output_ffn = ffn(X_with_pos)
print(f"Forma de salida FFN: {output_ffn.shape}")

---
## 6. Bloque Transformer Completo

Construimos un bloque encoder del transformer con:
- Multi-head attention
- Conexión residual
- Layer normalization
- Feed-forward network

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, num_heads)
        self.ffn = FeedForward(embed_dim, ff_dim, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Atención multi-head con conexión residual y normalización
        attn_out, attn_weights = self.attention(x)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Feed-forward con conexión residual y normalización
        ff_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ff_out))
        
        return x, attn_weights

# Crear bloque transformer
transformer_block = TransformerBlock(embed_dim=8, num_heads=2, ff_dim=16)

# Pasar datos a través del bloque
output_block, attn_block = transformer_block(X_with_pos)

print(f"Forma de salida del bloque: {output_block.shape}")
print(f"Forma de atención: {attn_block.shape}")

---
## 7. Encoder Transformer Completo

Apilamos varios bloques transformer para formar el encoder.

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, num_layers, max_len=100, dropout=0.1):
        super().__init__()
        self.pos_encoder = PositionalEncoding(embed_dim, max_len)
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Añadir positional encoding
        x = self.pos_encoder(x)
        x = self.dropout(x)
        
        # Pasar por cada bloque
        for layer in self.layers:
            x, attn = layer(x)
        
        return x

# Crear encoder con 3 capas
encoder = TransformerEncoder(embed_dim=8, num_heads=2, ff_dim=16, num_layers=3)

# Pasar datos a través del encoder
output_encoder = encoder(X_batch)
print(f"Forma de salida del encoder: {output_encoder.shape}")

---
## 8. Aplicación: Clasificación de Secuencias

Usamos el encoder transformer para clasificación (tomando el token CLS o promedio).

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, num_classes, max_len=100, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.encoder = TransformerEncoder(embed_dim, num_heads, ff_dim, num_layers, max_len, dropout)
        self.fc = nn.Linear(embed_dim, num_classes)
    
    def forward(self, x):
        # x: (batch, seq_len) con índices de palabras
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        encoded = self.encoder(embedded)  # (batch, seq_len, embed_dim)
        
        # Tomar el promedio de todas las posiciones (o usar el token CLS)
        pooled = encoded.mean(dim=1)  # (batch, embed_dim)
        
        return self.fc(pooled)

# Parámetros de ejemplo
vocab_size = 1000
num_classes = 2

classifier = TransformerClassifier(
    vocab_size=vocab_size,
    embed_dim=64,
    num_heads=4,
    ff_dim=128,
    num_layers=2,
    num_classes=num_classes
)

# Datos dummy de entrada
dummy_input = torch.randint(0, vocab_size, (4, 10))  # (batch=4, seq_len=10)
output = classifier(dummy_input)
print(f"Forma de salida del clasificador: {output.shape}")

---
## 9. Ejercicio para el Estudiante

1. **Atención causal (masked)**: Modifica la implementación de MultiHeadAttention para incluir una máscara causal (triangular inferior) que impida que las posiciones futuras atiendan a las pasadas. Esto es útil para decoders autoregresivos.

2. **Diferentes números de cabezas**: Experimenta con diferentes valores de `num_heads` y observa cómo cambian los patrones de atención.

3. **Visualiza las representaciones**: Toma las salidas del encoder y visualízalas con PCA o t-SNE para ver cómo se agrupan.

4. **Entrena un clasificador**: Usa el `TransformerClassifier` con un dataset real (ej. IMDb) y compara con LSTM.

In [ ]:
# Espacio para el estudiante
# ...

---
## 10. Conclusiones

En este notebook hemos implementado los componentes fundamentales del transformer:

✔️ **Self-Attention**: Calculamos matrices Q, K, V y pesos de atención.

✔️ **Multi-Head Attention**: Implementamos múltiples cabezas en paralelo que aprenden diferentes relaciones.

✔️ **Positional Encoding**: Añadimos información de posición usando funciones seno/coseno.

✔️ **Feed-Forward Network**: Capa MLP aplicada a cada posición.

✔️ **Bloque Transformer**: Integramos atención, residual, layer norm y FFN.

✔️ **Encoder Completo**: Apilamos bloques y añadimos positional encoding.

**Lección clave**: El transformer reemplaza las recurrencias con atención, permitiendo paralelización y mejor captura de dependencias largas. Es la base de modelos como BERT y GPT.

---
**Fin del Notebook Conceptual - Semana 9 (NLP)**